This script will clean the raw downloaded KSP incident unit datasets and exports the cleaned and combined dataset as a csv file and then uploads to the previously created SQLite database.

In [14]:
import os
import pandas as pd
import sqlite3

In [15]:
# Define the path for the SQLite database
cwd = os.getcwd()
database_path = f'{cwd}/data/crash_data.db'
os.makedirs(os.path.dirname(database_path), exist_ok=True)

# Define the path for the raw crash data files downloaded
directory_path = f'{cwd}/data/raw_crash_data'

In [16]:
# Function to check if tables created in the database exist
def check_table_exists(db_path, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to check if the table exists
    cursor.execute('''
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
    ''', (table_name,))

    # Fetch one record
    table_exists = cursor.fetchone() is not None

    # Close the connection
    conn.close()

    return table_exists

In [17]:
# Function to check if tables created in the database have data
def check_table_has_data(db_path, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to count the number of rows in the table
    cursor.execute(f'SELECT COUNT(*) FROM {table_name}')

    # Fetch the count
    row_count = cursor.fetchone()[0]

    # Close the connection
    conn.close()

    return row_count > 0


In [18]:
# Create/Connect to SQLite database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

In [19]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_factors (
        IncidentID INT,
        UnitNumber INT,
        Factor_Type TEXT,
        Factor TEXT);''')

In [20]:
# temp section to read the column names to create table columns

def read_column_names(csv_file_path):

    # Read only the first row of the CSV to get the column names
    df = pd.read_csv(csv_file_path, nrows=0)
    column_names = df.columns.tolist()
    return column_names

# Read the column names from the CSV file
csv_file_path = f'{cwd}/data/raw_crash_data/Unit_Factor_2024.csv'

columns = read_column_names(csv_file_path)
print(columns)


['IncidentId', 'UnitNumber', 'Factor_Type', 'Factor', 'Unnamed: 4']


In [21]:
# Check to see if tables created exist
table_name = 'ksp_factors'

if check_table_exists(database_path, table_name):
    print(f"The table '{table_name}' exists.")
else:
    print(f"The table '{table_name}' does not exist.")


The table 'ksp_factors' exists.


In [22]:
# Check of tables created are empty

table_name = 'ksp_factors'

if check_table_has_data(database_path, table_name):
    print(f"The table '{table_name}' contains data.")
else:
    print(f"The table '{table_name}' is empty.")


The table 'ksp_factors' is empty.


In [23]:

# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Unit_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_factors_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_factors_df.columns if col.startswith('Unnamed')]
combined_factors_df = combined_factors_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_factors_df)

All CSV files have been successfully loaded into a single DataFrame.
       IncidentId  UnitNumber           Factor_Type  Factor
0        32658708           1        ENVIRON FACTOR       2
1        32658708           2        ENVIRON FACTOR       2
2        32658708           3        ENVIRON FACTOR       2
3        32659228           1        ENVIRON FACTOR       2
4        32659228           2        ENVIRON FACTOR       2
...           ...         ...                   ...     ...
27277    30089216           1  DRIVER DISTRACTED BY       3
27278    30329814           1  DRIVER DISTRACTED BY       3
27279    29978338           1  DRIVER DISTRACTED BY       4
27280    30141594           1  DRIVER DISTRACTED BY       4
27281    29915633           1  DRIVER DISTRACTED BY       1

[27282 rows x 4 columns]


In [24]:

# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_factors.csv'

# Export the dataframe to a CSV file
combined_factors_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")


Dataframe exported successfully to /Users/terid/Git/CodeYou_Capstone/data/clean_crash_data/incident_factors.csv


In [25]:

# Write the DataFrame to the SQLite table
combined_factors_df.to_sql('ksp_factors', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()
conn.close()

print(f"Data from {df} has been successfully inserted into the ksp_factors table.")


Data from       IncidentId  UnitNumber           Factor_Type  Factor  Unnamed: 4
0       29540000           1        ENVIRON FACTOR       2         NaN
1       29540000           1        ENVIRON FACTOR       6         NaN
2       29540000           2        ENVIRON FACTOR      99         NaN
3       29556449           1        ENVIRON FACTOR       2         NaN
4       29556449           1        ENVIRON FACTOR      11         NaN
...          ...         ...                   ...     ...         ...
5416    30089216           1  DRIVER DISTRACTED BY       3         NaN
5417    30329814           1  DRIVER DISTRACTED BY       3         NaN
5418    29978338           1  DRIVER DISTRACTED BY       4         NaN
5419    30141594           1  DRIVER DISTRACTED BY       4         NaN
5420    29915633           1  DRIVER DISTRACTED BY       1         NaN

[5421 rows x 5 columns] has been successfully inserted into the ksp_factors table.
